In [2]:
import torch, torchio as tio
import torch.nn.functional as F

from torch.utils.data import DataLoader
from datasets import *
from models import *
from pathlib import Path
from torch.amp import autocast
from sklearn.model_selection import train_test_split

In [3]:
%load_ext autoreload
%autoreload 2

In [5]:
model = ConvNeXtV2Encoder(
    depths=[3, 3, 27, 3], dims=[128, 256, 512, 1024], proj_hidden_dim=2048
)
for i, (k, v) in enumerate(model.named_parameters()):
    print(f"{i:03d}  {k:45s}  shape={tuple(v.shape)}")
    if i >= 200 - 1:
        break

000  downsample_layers.0.0.weight                   shape=(128, 6, 4, 4, 4)
001  downsample_layers.0.0.bias                     shape=(128,)
002  downsample_layers.0.1.weight                   shape=(128,)
003  downsample_layers.0.1.bias                     shape=(128,)
004  downsample_layers.1.0.weight                   shape=(128,)
005  downsample_layers.1.0.bias                     shape=(128,)
006  downsample_layers.1.1.weight                   shape=(256, 128, 2, 2, 2)
007  downsample_layers.1.1.bias                     shape=(256,)
008  downsample_layers.2.0.weight                   shape=(256,)
009  downsample_layers.2.0.bias                     shape=(256,)
010  downsample_layers.2.1.weight                   shape=(512, 256, 2, 2, 2)
011  downsample_layers.2.1.bias                     shape=(512,)
012  downsample_layers.3.0.weight                   shape=(512,)
013  downsample_layers.3.0.bias                     shape=(512,)
014  downsample_layers.3.1.weight                   s

In [4]:
@torch.inference_mode()
def knn_self_accuracy(
    encoder, loader: DataLoader, k=5, device="cuda", max_batches=None
):
    encoder.eval()
    z1_list, z2_list = [], []

    for b, (x1, x2) in enumerate(loader, 1):
        if max_batches and b > max_batches:
            break
        x1 = x1.to(device, non_blocking=True)
        x2 = x2.to(device, non_blocking=True)

        with autocast(device.type):
            z1 = F.normalize(encoder(x1), dim=1)
            z2 = F.normalize(encoder(x2), dim=1)
        z1_list.append(z1)
        z2_list.append(z2)

    z1_all = torch.cat(z1_list, dim=0)
    z2_all = torch.cat(z2_list, dim=0)
    z = torch.cat([z1_all, z2_all], dim=0)

    sim = torch.mm(z, z.t())
    sim.fill_diagonal_(-1e9)

    _, idx = sim.topk(k, dim=1)  # (2N, k)

    N = z1_all.size(0)
    ar = torch.arange(2 * N, device=z.device)
    pos_idx = torch.where(ar < N, ar + N, ar - N)  # (2N,)

    hits = (idx == pos_idx.view(-1, 1)).any(dim=1)
    topk_acc = hits.float().mean().item()
    return topk_acc

In [5]:
home_dir = "/home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training"

input_file = f"{home_dir}/tensor_paths.txt"
with open(input_file, "r") as f:
    final_paths = [Path(line.strip()) for line in f if line.strip()]

print(f"Loaded {len(final_paths)} paths from {input_file}")

_, val_paths = train_test_split(final_paths, test_size=0.2, random_state=42)

Loaded 23031 paths from /home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training/tensor_paths.txt


In [6]:
val_dataset = TensorDataset(path_list=val_paths)
val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
encoder = ResidualSEEncoder(layers=(3, 4, 6, 3), num_channels=6)
encoder.to(device)
ckpt = torch.load("./checkpoints/dti_best.pth", map_location=device, weights_only=True)
encoder.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)

<All keys matched successfully>

In [8]:
top5 = knn_self_accuracy(encoder, val_loader, k=1, device=device, max_batches=64)
print(f"Self k-NN top-1 accuracy: {top5:.3f}")

/home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training/python3.10/lib/python3.10/site-packages/torchio/data/image.py:248: UserWarning: Using TorchIO images without a torchio.SubjectsLoader in PyTorch >= 2.3 might have unexpected consequences, e.g., the collated batches will be instances of torchio.Subject with 5D images. Replace your PyTorch DataLoader with a torchio.SubjectsLoader so that the collated batch becomes a dictionary, as expected. See https://github.com/TorchIO-project/torchio/issues/1179 for more context about this issue.
  warnings.warn(message, stacklevel=1)
/home-local/lij112/codes/beyond_fa_challenge/beyond_fa_infonce/tensor_metric_infonce_training/python3.10/lib/python3.10/site-packages/torchio/data/image.py:248: UserWarning: Using TorchIO images without a torchio.SubjectsLoader in PyTorch >= 2.3 might have unexpected consequences, e.g., the collated batches will be instances of torchio.Subject with 5D images. Replace your PyTorc

[Warning] File not found: /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA4568/ses-010scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA4572/ses-010scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA5331/ses-130scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA4721/ses-080scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA4626/ses-010scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/BLSA/derivatives/sub-BLSA1704/ses-130scanner12/PreQual/TENSOR/dwmri_tensor.nii.gz, trying next index...
[Warning] File not found: /nfs2/harmonization/BIDS/BLSA/derivatives/su